In [1]:
import torch
import pandas as pd
from pathlib import Path

from src.prompt_manager import PromptManager
from src.data_manager import DataManager

from src import data_processing
import yaml
from src import paths
import hashlib

from src.utils import interactive_dataframe_selector

In [ ]:
with open('../src/configs/config.yaml', 'r') as f:
    full_config = yaml.safe_load(f)

# active_analysis = full_config['active_analysis']
active_analysis = 'mcgill_qa_feedback'
model_vars = full_config['analyses'][active_analysis]['model_vars']
experimental_groups = model_vars['experimental_groups']

In [4]:
raw_df = pd.read_parquet(paths.RAW_DATA_DIR / f'{active_analysis}.parquet')

In [5]:
final_df = data_processing.get_analysis_ready_df(full_config=full_config,
                                                 active_analysis='mcgill_qa_feedback',
                                                 use_cache=False,
                                                 force_refresh=False)

Loading files for analysis mcgill_qa_feedback
🐢 Running full processing pipeline...
Finished loading experiment data
Found 0 experimental trials contaminated by garbage output.


C:\Users\Wouter Barter\Documents\AI_thesis\src\data_processing.py:278: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  weights_tensor = torch.tensor(weights, dtype=torch.float32)


In [7]:
from src.analysis.reliability import ReliabilityAnalyzer

In [8]:
analyzer = ReliabilityAnalyzer(final_df, group_cols=[
                               'model_name', 'prompt_id', 'dimension_name'], llm_rating_col='mean_rating')
analyzer.compute_reliability_gap(metric='spearman')

,model_name,prompt_id,dimension_name,metric_type,spearman_gap,spearman_hh,spearman_llm_avg
3,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Relevance,Spearman,-0.030612,0.633026,0.663638
2,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Directness,Spearman,-0.020008,0.633026,0.653035
1,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Completeness,Spearman,-0.016169,0.633026,0.649195
0,Qwen/Qwen3-4B-Instruct-2507,261acd58fe,quality,Spearman,0.006445,0.633026,0.626581


In [9]:
analyzer = ReliabilityAnalyzer(
    final_df, group_cols=['model_name', 'prompt_id', 'dimension_name'])
analyzer.analyze_calibration('human_disagreement')

,model_name,prompt_id,dimension_name,calibration_corr
0,Qwen/Qwen3-4B-Instruct-2507,261acd58fe,quality,0.306506
1,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Completeness,0.314166
2,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Directness,0.267166
3,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Relevance,0.282543


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import scipy.stats as stats

In [ ]:
full_model_df = final_df[['input_id', 'prompt_id', 'model_name', 'mean_human_rating',
                          'mode_rating', 'mean_rating', 'dimension_name', 'normalized_entropy']]

In [ ]:
pm = PromptManager(folder=Path(
    "../prompts/PromptSuites/sandbox/MCGILL_QA_FEEDBACK"))
pm.load_all()

PromptManager initialized with folder: ..\prompts\PromptSuites\sandbox\MCGILL_QA_FEEDBACK
Scanning 2 suites from ..\prompts\PromptSuites\sandbox\MCGILL_QA_FEEDBACK...
Loaded 2 PromptSuites


{'3fc3c2dce92c': PromptSuite(id='3fc3c2dce92c', templates={'Relevance': PromptTemplate(id='f67646daad', name='naive_Relevance', dimension_name='Relevance', description='Naive prompt that measures the Relevance dimension', token_constraints=['1', '2', '3', '4'], tags=['naive', 'Relevance', 'baseline', 'scale_4'], system_message='You are an expert evaluator. Your task is to rate the overall Relevance of the Answer provided for the Question on a scale of 1 to 4, where 1 is the lowest Relevance and 4 is the highest Relevance.', user_message_template='Question:\n    {question}\n\n    Answer: \n    {answer}', _cached_constraint_ids=None), 'Completeness': PromptTemplate(id='606d61a7f7', name='naive_Completeness', dimension_name='Completeness', description='Naive prompt that measures the Completeness dimension', token_constraints=['1', '2', '3', '4'], tags=['naive', 'Completeness', 'baseline', 'scale_4'], system_message='You are an expert evaluator. Your task is to rate the overall Completenes

In [13]:
pm.suites['261acd58fe'].tags

{'baseline', 'naive', 'quality', 'scale_4'}

In [ ]:
models = {}

for group_key, group_df in full_model_df.groupby(['model_name', 'prompt_id']):
    if group_df['dimension_name'].unique()[0] == 'holistic':
        formula = 'mean_human_rating ~ mean_rating'
        group_df_wide = group_df
    else:
        group_df_wide = group_df.pivot_table(index=['input_id', 'mean_human_rating', 'model_name'],
                                             columns='dimension_name',
                                             values=['normalized_entropy', 'mean_rating']).reset_index()
        group_df_wide.columns = ['_'.join(col).strip(
            '_') if col[1] else col[0] for col in group_df_wide.columns.values]
        formula = "mean_human_rating ~ mean_rating_relevance + mean_rating_completeness + mean_rating_directness"

    model = smf.ols(formula, data=group_df_wide).fit()

    models[str(f"{group_key[0]}_{prompt_hash_map[group_key[1]]}")] = model